# 02 — MCP Tools

Tools are **model-controlled** primitives — Claude decides when to use them.

## Learning Objectives

- Define MCP tools using the `@mcp.tool` decorator
- Use `Field()` for parameter descriptions
- Understand auto-generated JSON schemas
- Handle errors in tools

In [ ]:
# First, let's understand the structure of a tool definition
# We'll write tools as standalone code, then later run them in a server

# In-memory document store (simulating a real data source)
docs = {
    "report.pdf": "The report details the state of a 20m condenser tower.",
    "plan.md": "The plan outlines the steps for implementation.",
    "spec.txt": "Technical requirements for the equipment.",
    "financials.docx": "Budget and expenditure details for Q3.",
}

print(f"Document store has {len(docs)} documents:")
for doc_id in docs:
    print(f"  - {doc_id}")

## Tool 1: Read Document Contents

This tool takes a document ID and returns its contents.

In [ ]:
# Tool implementation (will be decorated with @mcp.tool in a server)

def read_doc_contents(doc_id: str) -> str:
    """Read the contents of a document and return it as a string."""
    if doc_id not in docs:
        raise ValueError(f"Doc with id '{doc_id}' not found")
    return docs[doc_id]

# Test it
print(read_doc_contents("report.pdf"))
print()

# Test error handling
try:
    read_doc_contents("nonexistent.txt")
except ValueError as e:
    print(f"Error handled: {e}")

## Tool 2: Edit Document

This tool performs a find-and-replace operation on a document.

In [ ]:
def edit_document(doc_id: str, old_str: str, new_str: str) -> str:
    """Edit a document by replacing an exact string with a new string."""
    if doc_id not in docs:
        raise ValueError(f"Doc with id '{doc_id}' not found")
    if old_str not in docs[doc_id]:
        raise ValueError(f"Text '{old_str}' not found in document '{doc_id}'")
    
    docs[doc_id] = docs[doc_id].replace(old_str, new_str)
    return f"Document '{doc_id}' updated successfully"

# Test editing
print("Before:", docs["report.pdf"])
result = edit_document("report.pdf", "20m", "20-meter")
print(result)
print("After:", docs["report.pdf"])

## Writing Tools with FastMCP

Here's how these tools look when defined in an MCP server:

In [ ]:
# This is the complete server code that would go in a .py file
SERVER_CODE = '''
from mcp.server.fastmcp import FastMCP
from pydantic import Field

mcp = FastMCP("DocumentMCP", log_level="ERROR")

docs = {
    "report.pdf": "The report details the state of a 20m condenser tower.",
    "plan.md": "The plan outlines the steps for implementation.",
}

@mcp.tool(
    name="read_doc_contents",
    description="Read the contents of a document and return it as a string.",
)
def read_document(
    doc_id: str = Field(description="Id of the document to read"),
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    return docs[doc_id]


@mcp.tool(
    name="edit_document",
    description="Edit a document by replacing a string with a new string.",
)
def edit_document(
    doc_id: str = Field(description="Id of the document to edit"),
    old_str: str = Field(description="Text to replace (must match exactly)"),
    new_str: str = Field(description="New text to insert"),
):
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    docs[doc_id] = docs[doc_id].replace(old_str, new_str)
    return f"Document {doc_id} updated"

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

print("Key decorators and patterns:")
print("  @mcp.tool(name=..., description=...) — registers the tool")
print("  Field(description=...) — describes each parameter")
print("  raise ValueError(...) — signals errors to the client")
print("  mcp.run(transport='stdio') — starts the server")

## Auto-Generated JSON Schema

FastMCP automatically generates a JSON schema from the function signature:

In [ ]:
import json

# This is what FastMCP generates automatically from the @mcp.tool decorator
auto_generated_schema = {
    "name": "read_doc_contents",
    "description": "Read the contents of a document and return it as a string.",
    "input_schema": {
        "type": "object",
        "properties": {
            "doc_id": {
                "type": "string",
                "description": "Id of the document to read",
            }
        },
        "required": ["doc_id"],
    },
}

print("Auto-generated tool schema:")
print(json.dumps(auto_generated_schema, indent=2))

## Exercise: Create Your Own Tool

Create a `search_documents` tool that:
1. Takes a `query` string parameter
2. Searches all documents for the query
3. Returns matching document IDs and relevant excerpts

In [ ]:
# TODO: Implement search_documents

def search_documents(query: str) -> str:
    """Search all documents for a keyword and return matches."""
    # Reset docs for exercise
    docs = {
        "report.pdf": "The report details the state of a 20-meter condenser tower.",
        "plan.md": "The plan outlines the steps for implementation.",
        "spec.txt": "Technical requirements for the equipment.",
        "financials.docx": "Budget and expenditure details for Q3.",
    }
    
    # Your implementation here:
    matches = []
    for doc_id, content in docs.items():
        if query.lower() in content.lower():
            matches.append(f"{doc_id}: ...{content}...")
    
    if not matches:
        return f"No documents contain '{query}'"
    return "\n".join(matches)

# Test
print(search_documents("report"))
print()
print(search_documents("budget"))
print()
print(search_documents("nonexistent"))